# Seaquest HF-Expert / Original Critic (retry)
Train the **unchanged** committed `seaquest_ccrl` contrastive critic on the HF-expert dataset (`raw_hf`) and run the action-use diagnostic. You upload `raw_hf` to Colab yourself and set `DATA_ROOT` in Cell 2. Only the dataset *source* differs from the original action-blind run.

**Use:** set the token (Cell 1) + `DATA_ROOT` (Cell 2), then Run-all. Training starts at Cell 3.

In [ ]:
import torch
print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '-')

In [ ]:
# 1. Clone the (private) repo with your token, enter it, pull latest.
TOKEN = 'PASTE_YOUR_GITHUB_TOKEN_HERE'   # PAT with repo read access
import os, subprocess
D = '/content/Goal-Conditioned-Confounded-MsPacman'
if not os.path.isdir(D):
    subprocess.run(['git','clone',f'https://{TOKEN}@github.com/tingrui-huang/Goal-Conditioned-Confounded-MsPacman.git',D], check=True)
%cd /content/Goal-Conditioned-Confounded-MsPacman
!git pull -q && git log --oneline -1
# numpy+torch are preinstalled; OCAtari is NOT needed (get_game lazily imports the env; the offline
# training + action-use diagnostic never touch the env).

In [ ]:
# 2. Point at the raw_hf dataset YOU uploaded to Colab (folder with traj_*.npz + manifest.json).
import glob
DATA_ROOT = '/content/raw_hf'      # <-- EDIT to wherever you put raw_hf
n = len(glob.glob(DATA_ROOT + '/traj_*.npz'))
print('DATA_ROOT =', DATA_ROOT, '| trajectories:', n)
assert n == 40, f'expected 40 traj at {DATA_ROOT}, found {n} -- check the path / upload'
# optional compatibility gate (should print PROCEED):
!python -m seaquest_ccrl.scripts.gate_hf_dataset --hf-root "$DATA_ROOT" --old-root seaquest_ccrl/data/raw

In [ ]:
# 3. Train the committed critic on raw_hf (seed 0, masked/naive learner view). GPU ~minutes.
!python -m seaquest_ccrl.scripts.run_hf_expert_critic --root "$DATA_ROOT" --steps 50000 --seed 0 --ckpt-dir seaquest_ccrl/checkpoints/hf_seed0

In [ ]:
# 4. Action-use diagnostic (evaluation only; trains nothing).
!python -m seaquest_ccrl.scripts.eval_hf_action_use --ckpt seaquest_ccrl/checkpoints/hf_seed0/critic_naive.pt --root "$DATA_ROOT" --out artifacts/seaquest/hf_original_critic/action_use_diag.json
print(open('artifacts/seaquest/hf_original_critic/action_use_diag.json').read())

In [ ]:
# 5. (optional) download checkpoint + diagnostic
import shutil, os
for f in ['seaquest_ccrl/checkpoints/hf_seed0/critic_naive.pt',
          'seaquest_ccrl/checkpoints/hf_seed0/history_naive.json']:
    if os.path.exists(f): shutil.copy(f, 'artifacts/seaquest/hf_original_critic/')
shutil.make_archive('hf_result', 'zip', 'artifacts/seaquest/hf_original_critic')
try:
    from google.colab import files; files.download('hf_result.zip')
except Exception: pass